# Fuel Poverty Economics Project

Analysis of sub-regional fuel poverty in England using the 2024 tables. The notebook cleans local-authority data, compares regional rates, identifies hotspots, demonstrates SQL queries, and fits exploratory regressions.

**Data required:** the UK government fuel-poverty workbook (`fuel-poverty-sub-regional-2026-2024-data-tables.xlsx`). Place it beside this notebook, or update `FILE_PATH` below.

> The income and EPC variables in the final regression section are simulated solely to demonstrate workflow. They are not observed data and must not be interpreted as evidence about real policy drivers.

In [ ]:
%pip install pandas numpy matplotlib seaborn statsmodels plotly duckdb openpyxl

import os
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns
import statsmodels.api as sm

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid')

FILE_PATH = Path('fuel-poverty-sub-regional-2026-2024-data-tables.xlsx')
OUTPUT_DIR = Path('data/processed')
VISUALISATION_DIR = Path('visualisations')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VISUALISATION_DIR.mkdir(parents=True, exist_ok=True)

if not FILE_PATH.exists():
    raise FileNotFoundError(f'{FILE_PATH} not found. Put the workbook beside this notebook or update FILE_PATH.')

## Load and clean local-authority data

In [ ]:
# Table 2 has a title row followed by the actual column labels.
df = pd.read_excel(FILE_PATH, sheet_name='Table 2', skiprows=1).dropna(how='all').reset_index(drop=True)

# Promote the first remaining row to column names and remove it from the observations.
df.columns = df.iloc[0].astype(str).str.strip()
df = df.iloc[1:].reset_index(drop=True)
df.columns = [
    'Local_Authority_Code', 'Region', 'County_or_Unitary_Authority',
    'Local_Authority_District', 'Number_of_Households',
    'Households_in_Fuel_Poverty', 'Fuel_Poverty_Percentage'
]

for column in ['Number_of_Households', 'Households_in_Fuel_Poverty', 'Fuel_Poverty_Percentage']:
    df[column] = pd.to_numeric(df[column], errors='coerce')

# Keep rows with a district name; this excludes England, regional totals, and county-level summaries.
district_df = df.loc[df['Local_Authority_District'].notna()].copy()
district_df['Region'] = district_df['Region'].astype('string').str.strip()
district_df['Local_Authority_District'] = district_df['Local_Authority_District'].astype('string').str.strip()

print(f'District observations: {len(district_df):,}')
display(district_df.head())
display(district_df[['Number_of_Households', 'Households_in_Fuel_Poverty', 'Fuel_Poverty_Percentage']].describe())

## Regional disparities and hotspot identification

In [ ]:
regional_summary = (
    district_df.groupby('Region', as_index=False)
    .agg(
        Total_Households=('Number_of_Households', 'sum'),
        Total_Fuel_Poor_Households=('Households_in_Fuel_Poverty', 'sum'),
        Average_Fuel_Poverty_Rate=('Fuel_Poverty_Percentage', 'mean'),
    )
)
regional_summary['Regional_Fuel_Poverty_Rate_Overall'] = (
    regional_summary['Total_Fuel_Poor_Households'] / regional_summary['Total_Households'] * 100
)
regional_summary = regional_summary.sort_values('Regional_Fuel_Poverty_Rate_Overall', ascending=False)
display(regional_summary)

# The top decile by local-authority rate; ceil ensures at least one observation when data is present.
n_hotspots = max(1, int(np.ceil(len(district_df) * 0.10)))
hotspots = district_df.nlargest(n_hotspots, 'Fuel_Poverty_Percentage')
display(hotspots[['Local_Authority_District', 'Region', 'Number_of_Households',
                   'Households_in_Fuel_Poverty', 'Fuel_Poverty_Percentage']])

cleaned_file = OUTPUT_DIR / 'cleaned_fuel_poverty_2024.csv'
district_df.to_csv(cleaned_file, index=False)
print(f'Saved cleaned district data to {cleaned_file}')

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(district_df['Fuel_Poverty_Percentage'].dropna(), bins=30, kde=True, color='skyblue')
plt.title('Distribution of Fuel Poverty Percentage')
plt.xlabel('Fuel poverty rate (%)')
plt.ylabel('Number of local authorities')
plt.tight_layout()
plt.show()

## SQL analysis with DuckDB

In [ ]:
con = duckdb.connect(':memory:')
con.register('fuel_poverty', district_df)
sql_query = '''
SELECT Local_Authority_District, Region, Number_of_Households,
       Households_in_Fuel_Poverty, Fuel_Poverty_Percentage
FROM fuel_poverty
ORDER BY Fuel_Poverty_Percentage DESC
LIMIT 10
'''
sql_results = con.execute(sql_query).df()
display(sql_results)

## Exploratory OLS regression
The following single-variable model is descriptive only. Household count is not a causal policy driver, and the observations are cross-sectional.

In [ ]:
model_data = district_df[['Fuel_Poverty_Percentage', 'Number_of_Households']].dropna()
X = sm.add_constant(model_data['Number_of_Households'])
Y = model_data['Fuel_Poverty_Percentage']
model = sm.OLS(Y, X).fit(cov_type='HC3')
print(model.summary())

## Simulated multivariable regression demonstration
**Do not interpret the results below as empirical findings.** Income and EPC values are generated from fuel-poverty rates in order to illustrate model syntax only. A substantive analysis requires observed covariates from documented sources, appropriate geographic matching, and a defensible research design.

In [ ]:
rng = np.random.default_rng(42)
demo_data = district_df[['Fuel_Poverty_Percentage']].dropna().copy()
rate = demo_data['Fuel_Poverty_Percentage'].to_numpy()
demo_data['Simulated_Median_Income_GBP'] = rng.normal(32000, 5000, len(demo_data)) - rate * 800
demo_data['Simulated_EPC_D_or_Below_Percentage'] = rng.normal(50, 10, len(demo_data)) + rate * 1.5
X_demo = sm.add_constant(demo_data[['Simulated_Median_Income_GBP', 'Simulated_EPC_D_or_Below_Percentage']])
demo_model = sm.OLS(demo_data['Fuel_Poverty_Percentage'], X_demo).fit(cov_type='HC3')
print(demo_model.summary())

## Interactive income and fuel-poverty scatterplot (simulated income)

In [ ]:
plot_data = district_df.copy()
# This is a simulated illustrative variable, not observed household income.
plot_data['Simulated_Median_Income_GBP'] = demo_data['Simulated_Median_Income_GBP']
fig = px.scatter(
    plot_data, x='Simulated_Median_Income_GBP', y='Fuel_Poverty_Percentage',
    size='Number_of_Households', color='Region', hover_name='Local_Authority_District',
    title='Illustration only: Fuel Poverty vs Simulated Median Income by District',
    labels={'Simulated_Median_Income_GBP': 'Simulated median household income (£)',
            'Fuel_Poverty_Percentage': 'Fuel poverty rate (%)'}, template='plotly_white'
)
fig.write_html(VISUALISATION_DIR / 'regional_inequality_scatter.html')
fig.show()

## Summary and limitations

Use the weighted regional rate (total fuel-poor households divided by total households) when comparing region-level prevalence; the unweighted mean of district rates answers a different question. Check the source workbook notes and geographic hierarchy before reporting results. The supplied workbook reports estimates for 2024; document the publication date and source in any report.

The exploratory OLS model does not establish causality. Do not report the simulated covariate regression as evidence. Replace simulated values with validated, observed data before making claims about income or housing efficiency.